# Pipeline de Extração de Manuais Militares — PDF para XML

Este notebook processa manuais de campanha do Exército Brasileiro em PDF, extraindo a estrutura completa (capítulos, seções, anexos) e exportando para arquivos XML estruturados.

**Objetivo:** Preparar documentos militares em formato XML para futuro treinamento de agentes de IA.

**Fluxo:**
1. Descobre automaticamente todos os PDFs na pasta `Arquivos/`
2. Extrai o título de cada documento (padrão EB na primeira página)
3. Parseia o índice de assuntos para identificar a estrutura (capítulos, seções, anexos)
4. Mapeia a numeração das páginas (formato `X-Y`) para índices reais do PDF
5. Extrai o texto de cada seção
6. Exporta para XML na pasta `Gerados/`

In [ ]:
import fitz
import os
import re
from pathlib import Path
from collections import OrderedDict
from lxml import etree

## 1. Configuração e Descoberta de Arquivos

Define as pastas de entrada e saída, e descobre automaticamente todos os arquivos `.pdf` presentes na pasta `Arquivos/`. Para adicionar um novo manual, basta colocar o PDF nessa pasta — não é necessário alterar o código.

In [ ]:
PASTA_ARQUIVOS = Path("Arquivos")
PASTA_GERADOS = Path("Gerados")
PASTA_GERADOS.mkdir(parents=True, exist_ok=True)

arquivos = {}
for pdf_path in sorted(PASTA_ARQUIVOS.glob("*.pdf")):
    nome = pdf_path.stem
    arquivos[nome] = str(pdf_path)

print(f"📂 PDFs encontrados em '{PASTA_ARQUIVOS}/':")
for nome, caminho in arquivos.items():
    print(f"  - {nome}: {caminho}")
print(f"\n📁 XMLs serão salvos em: '{PASTA_GERADOS}/'")

## 2. Extração de Títulos

Os manuais da EB seguem um padrão na primeira página: as linhas 4 a 7 contêm o comando, tipo de manual, título e edição. Esta função lê essas linhas e monta um título descritivo para cada documento, que será usado como nome do arquivo XML e da pasta de imagens.

In [ ]:
def extrair_titulo_pdf(caminho_pdf):
    pdf = fitz.open(str(caminho_pdf))
    texto_pagina_1 = pdf[0].get_text("text")
    pdf.close()

    linhas = []
    for linha in texto_pagina_1.splitlines():
        linha_limpa = re.sub(r"\s+", " ", linha).strip()
        if linha_limpa:
            linhas.append(linha_limpa)

    if len(linhas) >= 7:
        titulo = " | ".join(linhas[3:7])
    elif len(linhas) >= 4:
        titulo = " ".join(linhas[3:])
    else:
        titulo = " | ".join(linhas)

    titulo = re.sub(r"\s{2,}", " ", titulo).strip()
    return titulo


titulos = {}

for nome, caminho in arquivos.items():
    caminho_path = Path(caminho)
    if not caminho_path.exists():
        print(f"{nome}: arquivo não encontrado -> {caminho_path}")
        continue

    titulos[nome] = extrair_titulo_pdf(caminho_path)
    print(f"{nome}: {titulos[nome]}")

print("\nDicionário de títulos:")
print(titulos)

## 3. Funções de Extração e Processamento

Conjunto de funções que realizam o trabalho pesado do pipeline:

- **`sanitizar()`** — Remove caracteres inválidos para nomes de arquivo/pasta
- **`extrair_indice_pdf()`** — Localiza e extrai o texto do índice de assuntos (entre "ÍNDICE DE ASSUNTOS" e o início do Capítulo I)
- **`preprocessar_indice()`** — Normaliza o índice: junta linhas fragmentadas ("CAPÍTULO" solto, números de seção soltos, páginas soltas)
- **`parsear_indice()`** — Converte o índice em uma lista estruturada com tipo (capítulo/seção/anexo/especial), número, título e página
- **`mapear_paginas()`** — Cria o mapa entre a numeração do manual (ex: `3-1`) e o índice real da página no PDF
- **`_pag_range()`** — Calcula o intervalo de páginas de cada seção
- **`_limpar_texto()` / `_extrair_texto_paginas()`** — Extraem e limpam o texto (remove cabeçalhos EB, numeração de página solta)
- **`construir_documento()`** — Monta o dicionário aninhado com todo o conteúdo
- **`criar_pastas_e_imagens()`** — Cria a estrutura de pastas e extrai imagens do PDF

In [ ]:
base_dir = Path(".")


def sanitizar(nome, max_len=200):
    """Remove caracteres inválidos e trunca para nomes de pasta/arquivo."""
    nome = re.sub(r'[/\\:*?"<>|]', '', nome).strip()
    return nome[:max_len]


def _eh_conteudo_real(linha_strip):
    """Verifica se uma linha é header de conteúdo real (não entrada de índice)."""
    if '…' in linha_strip or '...' in linha_strip:
        return False
    m = re.match(r'^\d+\.\d+\s+([A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-ZÁÉÍÓÚÂÊÔÃÕÇ]+)\b', linha_strip)
    if m and len(m.group(1)) >= 3:
        return True
    return False


def extrair_indice_pdf(caminho_pdf):
    """Extrai o texto do índice entre 'ÍNDICE DE ASSUNTOS' e o CAPÍTULO I real."""
    pdf = fitz.open(str(caminho_pdf))
    indice = ""
    capturando = False

    for pagina in pdf:
        texto = pagina.get_text("text")
        if not capturando:
            if "NDICE DE ASSUNTOS" in texto:
                capturando = True
                idx = texto.find("NDICE DE ASSUNTOS")
                indice += texto[idx:] + "\n"
        else:
            linhas = texto.splitlines()
            parar = False
            for linha in linhas:
                if _eh_conteudo_real(linha.strip()):
                    parar = True
                    break
            if parar:
                break
            indice += texto + "\n"

    pdf.close()
    return indice


def preprocessar_indice(indice_texto):
    """Normaliza o texto do índice."""
    linhas = indice_texto.splitlines()

    linhas_juntas = []
    i = 0
    while i < len(linhas):
        ls = linhas[i].strip()
        if not ls:
            i += 1
            continue
        if re.match(r'^CAP[ÍI]TULO$', ls) and i + 1 < len(linhas):
            proxima = linhas[i + 1].strip()
            linhas_juntas.append(ls + " " + proxima)
            i += 2
            continue
        if re.match(r'^\d+\.\d+$', ls) and i + 1 < len(linhas):
            proxima = linhas[i + 1].strip()
            linhas_juntas.append(ls + " " + proxima)
            i += 2
            continue
        linhas_juntas.append(ls)
        i += 1

    resultado = []
    for ls in linhas_juntas:
        if ls in ('Pag', 'Pág', 'PAG'):
            continue
        if re.match(r'^\d+-\d+$', ls) and resultado:
            resultado[-1] += " " + ls
            continue
        eh_nova = bool(
            re.match(r'CAP[ÍI]TULO\s+[IVXLCDM]+', ls) or
            re.match(r'\d+\.\d+\s+', ls) or
            re.match(r'ANEXO\s+[A-Z]', ls) or
            re.match(r'^(PREF[ÁA]CIO|GLOSS[ÁA]RIO|REFER[ÊE]NCIAS)', ls, re.IGNORECASE) or
            "NDICE DE ASSUNTOS" in ls
        )
        if eh_nova:
            resultado.append(ls)
        elif resultado:
            resultado[-1] += " " + ls

    return "\n".join(resultado)


def parsear_indice(indice_texto):
    """Parse o índice em lista de dicts: {tipo, num, titulo, pagina}."""
    indice_texto = preprocessar_indice(indice_texto)
    estrutura = []

    for linha in indice_texto.splitlines():
        linha = linha.strip()
        if not linha or "NDICE DE ASSUNTOS" in linha:
            continue

        m = re.match(r'CAP[ÍI]TULO\s+([IVXLCDM]+)\s*[–\-—]\s*(.+)', linha)
        if m:
            titulo = re.sub(r'[\.\…]+[\s\d\-]*$', '', m.group(2)).strip()
            pag_m = re.search(r'(\d+-\d+)\s*$', linha)
            estrutura.append({
                'tipo': 'capitulo', 'num': m.group(1),
                'titulo': titulo, 'pagina': pag_m.group(1) if pag_m else None
            })
            continue

        m = re.match(r'(\d+\.\d+)\s+(.+)', linha)
        if m:
            num = m.group(1)
            resto = m.group(2)
            pag_m = re.search(r'(\d+-\d+)\s*$', resto)
            titulo = re.sub(r'[\.\…]+[\s\d\-]*$', '', resto).strip()
            if titulo:
                estrutura.append({
                    'tipo': 'secao', 'num': num,
                    'titulo': titulo, 'pagina': pag_m.group(1) if pag_m else None
                })
            continue

        m = re.match(r'ANEXO\s+([A-Z])\s*[–\-—]\s*(.+)', linha)
        if m:
            titulo = re.sub(r'[\.\…]+[\s\d\-]*$', '', m.group(2)).strip()
            estrutura.append({
                'tipo': 'anexo', 'num': f"Anexo {m.group(1)}",
                'titulo': titulo, 'pagina': None
            })
            continue

        for p in ['PREFÁCIO', 'PREFACIO', 'GLOSSÁRIO', 'GLOSSARIO',
                   'REFERÊNCIAS', 'REFERENCIAS']:
            if linha.upper().startswith(p):
                titulo = p.title()
                pag_m = re.search(r'(\d+-\d+)\s*$', linha)
                estrutura.append({
                    'tipo': 'especial', 'num': '',
                    'titulo': titulo, 'pagina': pag_m.group(1) if pag_m else None
                })
                break

    return estrutura


def mapear_paginas(caminho_pdf):
    """Mapeia numeração do manual (X-Y) → índice de página do PDF."""
    pdf = fitz.open(str(caminho_pdf))
    mapa = {}
    for i, pagina in enumerate(pdf):
        texto = pagina.get_text("text")
        for linha in reversed(texto.splitlines()):
            ls = linha.strip()
            m = re.match(r'^(\d+-\d+)$', ls)
            if m:
                mapa[m.group(1)] = i
                break
    pdf.close()
    return mapa


def _pag_range(idx, estrutura, mapa_paginas, pdf_len):
    """Retorna (pag_inicio, pag_fim) para o item no índice idx."""
    item = estrutura[idx]
    pag_inicio = mapa_paginas.get(item.get('pagina')) if item.get('pagina') else None
    if pag_inicio is None:
        return None, None

    pag_fim = None
    for prox in estrutura[idx + 1:]:
        pp = mapa_paginas.get(prox.get('pagina')) if prox.get('pagina') else None
        if pp is not None and pp > pag_inicio:
            pag_fim = pp
            break
    if pag_fim is None:
        pag_fim = min(pag_inicio + 20, pdf_len)

    return pag_inicio, pag_fim


def _limpar_texto(texto):
    """Remove cabeçalhos de página e artefatos."""
    texto = re.sub(r'EB\d{2}-MC-\d{2}\.\d+', '', texto)
    texto = re.sub(r'^\s*\d+-\d+\s*$', '', texto, flags=re.MULTILINE)
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()


def _extrair_texto_paginas(pdf, pag_inicio, pag_fim):
    """Extrai e limpa o texto de um range de páginas."""
    texto = ""
    for p in range(pag_inicio, pag_fim):
        texto += pdf[p].get_text("text") + "\n"
    return _limpar_texto(texto)


def construir_documento(caminho_pdf, estrutura, mapa_paginas):
    """Constrói dicionário aninhado: {capitulo_titulo: {secao_titulo: texto}}."""
    pdf = fitz.open(str(caminho_pdf))
    documento = OrderedDict()
    cap_atual = None

    for idx, item in enumerate(estrutura):
        if item['tipo'] == 'capitulo':
            cap_atual = item['titulo']
            documento[cap_atual] = OrderedDict()

        elif item['tipo'] == 'secao':
            pag_i, pag_f = _pag_range(idx, estrutura, mapa_paginas, len(pdf))
            if pag_i is not None:
                texto = _extrair_texto_paginas(pdf, pag_i, pag_f)
                if cap_atual and cap_atual in documento:
                    documento[cap_atual][item['titulo']] = texto
                else:
                    documento[item['titulo']] = texto
            else:
                if cap_atual and cap_atual in documento:
                    documento[cap_atual][item['titulo']] = "(Página não localizada)"

        elif item['tipo'] in ('anexo', 'especial'):
            pag_i, pag_f = _pag_range(idx, estrutura, mapa_paginas, len(pdf))
            if pag_i is not None:
                texto = _extrair_texto_paginas(pdf, pag_i, pag_f)
                documento[item['titulo']] = texto
            else:
                documento[item['titulo']] = "(Conteúdo não extraído)"

    pdf.close()
    return documento


def criar_pastas_e_imagens(caminho_pdf, titulo_doc, estrutura, mapa_paginas, base_dir):
    """Cria estrutura de pastas e exporta imagens para cada seção/subseção."""
    pdf = fitz.open(str(caminho_pdf))
    pasta_doc = base_dir / sanitizar(titulo_doc)
    pasta_doc.mkdir(parents=True, exist_ok=True)

    cap_pasta = None
    total_imgs = 0

    for idx, item in enumerate(estrutura):
        if item['tipo'] == 'capitulo':
            nome = sanitizar(f"CAPÍTULO {item['num']} - {item['titulo']}")
            cap_pasta = pasta_doc / nome
            cap_pasta.mkdir(parents=True, exist_ok=True)

        elif item['tipo'] == 'secao':
            nome = sanitizar(f"{item['num']} {item['titulo']}")
            pasta_secao = (cap_pasta / nome) if cap_pasta else (pasta_doc / nome)
            pasta_secao.mkdir(parents=True, exist_ok=True)

            pag_i, pag_f = _pag_range(idx, estrutura, mapa_paginas, len(pdf))
            if pag_i is not None:
                for p in range(pag_i, pag_f):
                    for j, img_info in enumerate(pdf[p].get_images(full=True)):
                        xref = img_info[0]
                        try:
                            pix = fitz.Pixmap(pdf, xref)
                            if pix.n > 4:
                                pix = fitz.Pixmap(fitz.csRGB, pix)
                            img_path = pasta_secao / f"pagina_{p+1}_img_{j+1}.png"
                            pix.save(str(img_path))
                            total_imgs += 1
                        except Exception:
                            pass

        elif item['tipo'] in ('anexo', 'especial'):
            nome = sanitizar(item['titulo'])
            (pasta_doc / nome).mkdir(parents=True, exist_ok=True)

    pdf.close()
    return pasta_doc, total_imgs

## 4. Exportação para XML

Converte a estrutura extraída do PDF em um arquivo XML com tags semânticas, usando a biblioteca `lxml`. Cada documento gera um arquivo `.xml` na pasta `Gerados/`.

**Estrutura do XML gerado:**
```xml
<documento>
  <metadados>
    <titulo>COMANDO DE OPERAÇÕES TERRESTRES | Manual de Campanha | ORDEM UNIDA | 4ª Edição</titulo>
    <arquivo>doc1.pdf</arquivo>
  </metadados>
  <capitulo numero="I" titulo="INTRODUÇÃO">
    <secao numero="1.1" titulo="Finalidade">
      <texto>conteúdo extraído...</texto>
    </secao>
  </capitulo>
  <anexo letra="A" titulo="...">
    <texto>conteúdo...</texto>
  </anexo>
  <secao_especial titulo="Prefácio">
    <texto>conteúdo...</texto>
  </secao_especial>
</documento>
```

In [ ]:
def exportar_para_xml(caminho_pdf, titulo_doc, estrutura, mapa_paginas, pasta_saida):
    """Exporta o conteúdo do documento para um arquivo XML estruturado."""
    pdf = fitz.open(str(caminho_pdf))

    root = etree.Element("documento")

    meta = etree.SubElement(root, "metadados")
    etree.SubElement(meta, "titulo").text = titulo_doc
    etree.SubElement(meta, "arquivo").text = Path(caminho_pdf).name

    cap_node = None

    for idx, item in enumerate(estrutura):
        if item['tipo'] == 'capitulo':
            cap_node = etree.SubElement(root, "capitulo")
            cap_node.set("numero", item['num'])
            cap_node.set("titulo", item['titulo'])

        elif item['tipo'] == 'secao':
            pag_i, pag_f = _pag_range(idx, estrutura, mapa_paginas, len(pdf))
            secao_node = etree.SubElement(cap_node if cap_node is not None else root, "secao")
            secao_node.set("numero", item['num'])
            secao_node.set("titulo", item['titulo'])

            texto_node = etree.SubElement(secao_node, "texto")
            if pag_i is not None:
                texto_node.text = _extrair_texto_paginas(pdf, pag_i, pag_f)
            else:
                texto_node.text = "(Página não localizada)"

        elif item['tipo'] == 'anexo':
            pag_i, pag_f = _pag_range(idx, estrutura, mapa_paginas, len(pdf))
            anexo_node = etree.SubElement(root, "anexo")
            letra_match = re.search(r'[A-Z]$', item.get('num', ''))
            anexo_node.set("letra", letra_match.group(0) if letra_match else "")
            anexo_node.set("titulo", item['titulo'])

            texto_node = etree.SubElement(anexo_node, "texto")
            if pag_i is not None:
                texto_node.text = _extrair_texto_paginas(pdf, pag_i, pag_f)
            else:
                texto_node.text = "(Conteúdo não extraído)"

        elif item['tipo'] == 'especial':
            pag_i, pag_f = _pag_range(idx, estrutura, mapa_paginas, len(pdf))
            esp_node = etree.SubElement(root, "secao_especial")
            esp_node.set("titulo", item['titulo'])

            texto_node = etree.SubElement(esp_node, "texto")
            if pag_i is not None:
                texto_node.text = _extrair_texto_paginas(pdf, pag_i, pag_f)
            else:
                texto_node.text = "(Conteúdo não extraído)"

    pdf.close()

    tree = etree.ElementTree(root)
    nome_arquivo = sanitizar(titulo_doc) + ".xml"
    caminho_xml = pasta_saida / nome_arquivo
    tree.write(
        str(caminho_xml),
        encoding="UTF-8",
        xml_declaration=True,
        pretty_print=True
    )

    return caminho_xml

## 5. Execução do Pipeline

Para cada PDF descoberto na pasta `Arquivos/`, o pipeline executa as seguintes etapas em sequência:

1. **Extrai o índice** de assuntos do PDF
2. **Parseia** o índice em uma lista estruturada de capítulos, seções e anexos
3. **Mapeia** a numeração das páginas (ex: `3-1`) para os índices reais do PDF
4. **Constrói** um dicionário com o conteúdo completo do documento
5. **Cria pastas** organizadas por capítulo/seção e **extrai imagens** embutidas
6. **Exporta** o conteúdo completo para XML na pasta `Gerados/`

> Execute todas as células acima antes de rodar esta. Para adicionar novos manuais, basta colocar o PDF em `Arquivos/` e re-executar o notebook.

In [ ]:
docs_estrutura = {}

for nome, caminho in arquivos.items():
    caminho = Path(caminho)
    if not caminho.exists():
        print(f"⚠️ {nome}: não encontrado")
        continue

    print(f"\n{'='*60}")
    print(f"📄 Processando: {nome}")
    print(f"{'='*60}")

    indice = extrair_indice_pdf(caminho)
    if not indice:
        print(f"  ⚠️ Índice não encontrado")
        continue

    estrutura = parsear_indice(indice)
    mapa = mapear_paginas(caminho)

    cap_count = sum(1 for i in estrutura if i['tipo'] == 'capitulo')
    sec_count = sum(1 for i in estrutura if i['tipo'] == 'secao')
    print(f"  📋 {cap_count} capítulos, {sec_count} seções")
    print(f"  🗺️  {len(mapa)} páginas mapeadas")

    for item in estrutura:
        if item['tipo'] == 'capitulo':
            print(f"    📖 CAPÍTULO {item['num']} – {item['titulo']}")
        elif item['tipo'] == 'secao':
            print(f"       📄 {item['num']} {item['titulo']} (pág: {item['pagina']})")
        else:
            print(f"       📌 {item['tipo']}: {item['titulo']}")

    documento = construir_documento(caminho, estrutura, mapa)
    docs_estrutura[nome] = documento

    titulo = titulos.get(nome, nome)
    pasta, n_imgs = criar_pastas_e_imagens(caminho, titulo, estrutura, mapa, base_dir)
    print(f"  📁 Pastas criadas: {pasta}")
    print(f"  🖼️  {n_imgs} imagens exportadas")

    caminho_xml = exportar_para_xml(caminho, titulo, estrutura, mapa, PASTA_GERADOS)
    print(f"  📝 XML exportado: {caminho_xml}")


print(f"\n{'='*60}")
print(f"✅ Processamento concluído!")
print(f"{'='*60}")
print(f"\n📁 Arquivos XML gerados em '{PASTA_GERADOS}/':")
for xml_file in sorted(PASTA_GERADOS.glob("*.xml")):
    tamanho_kb = xml_file.stat().st_size / 1024
    print(f"  📝 {xml_file.name} ({tamanho_kb:.1f} KB)")